# Kokoro TTS: Generate Audio from Text

Synthesise speech from text using the [`kokoro`](https://pypi.org/project/kokoro/) library and [Kokoro-82M](https://huggingface.co/hexgrad/Kokoro-82M).

> **Before running**: follow the setup instructions in [`README.md`](./README.md) to create a dedicated `uv` virtual environment with all required packages.

Run cells top to bottom. Edit the **Configuration** cell to change the text, voice, or language.

## Imports & Device Detection

Loads required packages and selects the best available compute device (CUDA → MPS → CPU).

In [1]:
import os
import re
import time
import torch
import numpy as np
import soundfile as sf
import ipywidgets as widgets
from IPython.display import display, Audio

from kokoro import KPipeline

if torch.cuda.is_available():
    DEVICE = 'cuda'
elif os.environ.get('PYTORCH_ENABLE_MPS_FALLBACK') == '1' and torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

print(f'Using device: {DEVICE}')
print(f'Torch version: {torch.__version__}')

Using device: cuda
Torch version: 2.12.0+cu130


## Configuration

Edit the variables below to change the voice, language, speed, and text. All downstream cells read from these values unchanged.

In [2]:
# 'a' => American English   'b' => British English
LANG_CODE = 'a'

voice_widget = widgets.Dropdown(
    options=[
        ('af_sky — American Female',     'af_sky'),
        ('af_heart — American Female',   'af_heart'),
        ('af_bella — American Female',   'af_bella'),
        ('af_sarah — American Female',   'af_sarah'),
        ('af_nicole — American Female',  'af_nicole'),
        ('am_michael — American Male',   'am_michael'),
        ('am_adam — American Male',      'am_adam'),
        ('am_echo — American Male',      'am_echo'),
        ('am_eric — American Male',      'am_eric'),
        ('am_liam — American Male',      'am_liam'),
        
        ('bf_emma — British Female',     'bf_emma'),
        ('bf_isabella — British Female', 'bf_isabella'),
        ('bm_george — British Male',     'bm_george'),
        ('bm_lewis — British Male',      'bm_lewis'),
    ],
    value='af_sky',
    description='Voice:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='320px'),
)
display(voice_widget)

Dropdown(description='Voice:', layout=Layout(width='320px'), options=(('af_sky — American Female', 'af_sky'), …

In [3]:
pipeline = KPipeline(lang_code=LANG_CODE, repo_id='hexgrad/Kokoro-82M')

In [4]:
# 1.0 = normal, < 1.0 = slower, > 1.0 = faster
SPEED = 1.0
# Split on blank lines. Change to r'(?<=[.!?]) +' for sentence-level splits.
SPLIT_PATTERN = r'\n+'

# Path to a markdown file to narrate. If set, overrides TEXT below.
MARKDOWN_FILE = ''

TEXT = """
The open-source movement has transformed software development by enabling
collaboration across borders and disciplines. When developers share their
work freely, entire communities can build upon it — accelerating progress
in ways that no single team could achieve alone.
"""

OUTPUT_DIR = 'outputs'
OUTPUT_FILENAME = 'my_audio'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Config ready — voice: {voice_widget.value!r}, lang: {LANG_CODE!r}, speed: {SPEED}, output: {OUTPUT_DIR}/')

Config ready — voice: 'af_sky', lang: 'a', speed: 1.0, output: outputs/


## Text Source

When `MARKDOWN_FILE` is set, the title is pulled from the YAML frontmatter and the body prose is extracted automatically — code blocks, HTML/SVG, tables, blockquotes, and links are all removed. When `MARKDOWN_FILE` is empty, the inline `TEXT` variable is used instead.

In [5]:
def markdown_to_narration(path):
    """Return (title, body) from a markdown file, cleaned for TTS narration."""
    with open(path, encoding='utf-8') as f:
        raw = f.read()

    # 1. Split YAML frontmatter
    title = ''
    fm_match = re.match(r'^---\n(.*?)\n---\n', raw, re.DOTALL)
    if fm_match:
        fm = fm_match.group(1)
        t = re.search(r'^title:\s*["\']?(.*?)["\']?\s*$', fm, re.MULTILINE)
        if t:
            title = t.group(1).strip().strip('"\'')
        body = raw[fm_match.end():]
    else:
        body = raw

    # 2. Remove fenced code blocks (``` ... ```)
    body = re.sub(r'```[\s\S]*?```', '', body)

    # 3. Remove HTML/SVG blocks and leftover tags
    body = re.sub(r'<div[\s\S]*?</div>', '', body, flags=re.IGNORECASE)
    body = re.sub(r'<svg[\s\S]*?</svg>', '', body, flags=re.IGNORECASE)
    body = re.sub(r'<[^>]+>', '', body)

    # 4. Drop table lines and blockquote lines
    lines = body.splitlines()
    lines = [l for l in lines if not re.match(r'^\s*\|', l) and not re.match(r'^\s*>', l)]
    body = '\n'.join(lines)

    # 5. Drop horizontal rules
    body = re.sub(r'^\s*(-{3,}|\*{3,}|_{3,})\s*$', '', body, flags=re.MULTILINE)

    # 6. Remove images and autolinks
    body = re.sub(r'!\[[^\]]*\]\([^)]*\)', '', body)
    body = re.sub(r'<https?://[^>]+>', '', body)

    # 7. Drop inline links entirely (including visible text)
    body = re.sub(r'\[[^\]]*\]\([^)]*\)', '', body)

    # 8. Strip heading markers, keep text
    body = re.sub(r'^#{1,6}\s+', '', body, flags=re.MULTILINE)

    # 9. Strip list markers, emphasis, and inline-code backticks
    body = re.sub(r'^\s*[-*+]\s+', '', body, flags=re.MULTILINE)
    body = re.sub(r'^\s*\d+\.\s+', '', body, flags=re.MULTILINE)
    body = re.sub(r'\*{1,3}([^*]+)\*{1,3}', r'\1', body)
    body = re.sub(r'`([^`]+)`', r'\1', body)

    # 10. Strip stray non-ASCII symbols (emoji, dingbats) that don't narrate
    body = re.sub(r'[^\x00-\x7F]', '', body)

    # 11. Collapse 3+ blank lines to a double newline
    body = re.sub(r'\n{3,}', '\n\n', body).strip()

    return title, body


if MARKDOWN_FILE:
    title, body = markdown_to_narration(MARKDOWN_FILE)
    narration_text = f'{title}.\n\n{body}'
    base_name = os.path.splitext(os.path.basename(MARKDOWN_FILE))[0]
else:
    narration_text = TEXT.strip()
    base_name = OUTPUT_FILENAME

print(f'{len(narration_text)} chars to narrate — base name: {base_name!r}')

270 chars to narrate — base name: 'my_audio'


## Generate Audio

Initialises the synthesis pipeline and streams synthesis chunk by chunk. On first run, model weights (~327 MB) are downloaded from Hugging Face and cached locally. Saves the complete file and plays it inline when done.

In [6]:
SAMPLE_RATE = 24_000
audio_chunks = []

# ── progress UI ──────────────────────────────────────────────────────────────
bar = widgets.IntProgress(
    value=0, min=0, max=10,
    bar_style='info',
    layout=widgets.Layout(width='100%', height='28px'),
)
label = widgets.HTML(
    value='<span style="font-family:monospace;font-size:13px">⏳ &nbsp;Starting…</span>'
)
display(widgets.VBox([bar, label], layout=widgets.Layout(margin='8px 0')))
# ─────────────────────────────────────────────────────────────────────────────

t0 = time.perf_counter()
accumulated = 0.0

for i, (graphemes, phonemes, audio) in enumerate(pipeline(
    narration_text,
    voice=voice_widget.value,
    speed=SPEED,
    split_pattern=SPLIT_PATTERN,
)):
    if audio is None:
        continue
    audio_chunks.append(audio)
    accumulated += len(audio) / SAMPLE_RATE
    elapsed = time.perf_counter() - t0
    bar.max = max(bar.max, i + 2)   # always leave headroom — never hits 100% mid-run
    bar.value = i + 1
    label.value = (
        f'<span style="font-family:monospace;font-size:13px">'
        f'chunk&nbsp;<b>{i + 1:03d}</b>'
        f'&nbsp;&nbsp;|&nbsp;&nbsp;{accumulated:.1f}s audio synthesised'
        f'&nbsp;&nbsp;|&nbsp;&nbsp;{elapsed:.0f}s elapsed'
        f'</span>'
    )

full_audio = np.concatenate(audio_chunks)
total_dur = len(full_audio) / SAMPLE_RATE
elapsed = time.perf_counter() - t0
full_path = os.path.join(OUTPUT_DIR, f'{voice_widget.value}_{base_name}.wav')
sf.write(full_path, full_audio, SAMPLE_RATE)

bar.value = bar.max
bar.bar_style = 'success'
label.value = (
    f'<span style="font-family:monospace;font-size:13px;color:#2d8a4e">'
    f'<b>Done</b>'
    f'&nbsp;&nbsp;|&nbsp;&nbsp;{len(audio_chunks)} chunks'
    f'&nbsp;&nbsp;|&nbsp;&nbsp;{total_dur:.1f}s audio'
    f'&nbsp;&nbsp;|&nbsp;&nbsp;{elapsed:.1f}s wall time'
    f'&nbsp;&nbsp;|&nbsp;&nbsp;saved: <code>{full_path}</code>'
    f'</span>'
)

display(Audio(data=full_audio, rate=SAMPLE_RATE))